In [1]:
import pandas as pd

In [3]:
categorical_features = [
    "employment_type",
    "product_category",
    "location",
    "transaction_month",
    "transaction_dayofweek",
]


In [4]:
numeric_features = [
    "age",
    "monthly_income",
    "credit_score",
    "purchase_amount",
    "bnpl_installments",
    "app_usage_frequency",
    "debt_to_income_ratio",
    "transaction_year",
    "installment_amount",
    "purchase_to_income_ratio",
    "installment_to_income_ratio",
    "income_after_installment",
]


In [5]:
DROP_COLUMNS = [
    "user_id",
    "repayment_delay_days",
    "risk_score",
    "customer_segment",
]


In [6]:
df = pd.read_parquet("../data/processed/03_engineered_data.parquet")

In [7]:
def apply_split_reference(main_df: pd.DataFrame, ref_path: str, id_col: str):
    """
    Loads the split ledger and returns separated train and test DataFrames.
    """
    # Load the 2-column reference file
    split_ledger = pd.read_parquet(ref_path)
    
    # Merge the split assignment onto your main dataset
    # Using an inner merge ensures we only keep IDs that were officially split
    merged_df = main_df.merge(split_ledger, on=id_col, how='inner')
    
    # Separate the data
    train_df = merged_df[merged_df['split'] == 'train'].drop(columns=['split'])
    test_df = merged_df[merged_df['split'] == 'test'].drop(columns=['split'])
    
    return train_df, test_df

# Example Usage:
# train_data, test_data = apply_split_reference(main_df=my_engineered_data, ref_path='./data/metadata/split_ledger.parquet', id_col='customer_id')

In [9]:
train_data, test_data = apply_split_reference(
    main_df=df,
    ref_path="../data/metadata/split_ledger.parquet",
    id_col="user_id",
)

In [10]:
TARGET = "default_flag"

X_train = train_data.drop(
    columns=[TARGET] + DROP_COLUMNS
)

y_train = train_data[TARGET]


In [11]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ]
)


In [12]:
from sklearn.preprocessing import OneHotEncoder


categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        ),
    ]
)


In [13]:
from sklearn.compose import ColumnTransformer


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        ),
    ]
)


In [16]:
import joblib

joblib.dump(preprocessor, "../models/preprocessor/linear_preprocessor.joblib")

['../models/preprocessor/linear_preprocessor.joblib']

In [17]:
numeric_tree_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)


In [19]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_tree_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        ),
    ]
)

In [20]:
joblib.dump(tree_preprocessor, "../models/preprocessor/tree_preprocessor.joblib")

['../models/preprocessor/tree_preprocessor.joblib']